# 08 - Model comparison (Citi Bike hourly demand)

**Who worked on this:** *(fill in)*

One notebook that gathers the metrics of every model and compares them in depth, ending with our conclusions.

**Question (same for every model):** how many trips will start at a station in a given hour?
**Split:** chronological. Train = everything before the last 24 months, validation = the 12 months before the test period, test = the last 12 months.
**Rule:** models are ranked on **validation** only. The test columns are reported once and never used to choose.

In [ ]:
import sys, json
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sys.path.append(".")
from src.features import (load_demand, time_split, regression_metrics, load_metrics, seasonal_naive,
                          add_cluster_feature, load_cluster_mapping, TARGET)
sns.set_theme(style="whitegrid")

## 1. All models in one table
`load_metrics()` reads `models/results.json`, which every model notebook (04, 05, 07 and the SageMaker run) fills with `save_metrics`. Sorted by validation MAE.

In [ ]:
res = load_metrics()
cols = ["where", "valid_MAE", "valid_RMSE", "valid_R2", "test_MAE", "test_RMSE", "test_R2"]
for c in cols[1:]:
    res[c] = pd.to_numeric(res[c])
display(res[cols].round(3))
best = res.index[0]
print("Best on validation MAE:", best)

## 2. Validation vs test
A model that is much worse on test than on validation does not generalise to the future (or was tuned too hard on validation).

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 0.55 * len(res) + 2.5))
y = np.arange(len(res))
ax[0].barh(y - 0.2, res["valid_MAE"], 0.4, label="validation")
ax[0].barh(y + 0.2, res["test_MAE"], 0.4, label="test")
ax[0].set_yticks(y); ax[0].set_yticklabels(res.index); ax[0].invert_yaxis()
ax[0].set_xlabel("MAE (trips per hour), lower is better"); ax[0].legend(); ax[0].set_title("Error per model")
gap = (res["test_MAE"] - res["valid_MAE"])
ax[1].barh(y, gap, color=np.where(gap > 0, "tab:red", "tab:green"))
ax[1].set_yticks(y); ax[1].set_yticklabels(res.index); ax[1].invert_yaxis()
ax[1].set_xlabel("test MAE - validation MAE"); ax[1].set_title("Generalisation gap")
plt.tight_layout(); plt.show()

## 3. Improvement over the baselines
How much better is each model than the simplest reference (global mean) and than the best baseline? Percent reduction of validation MAE.

In [ ]:
ref = res["valid_MAE"]
base_name = [n for n in res.index if "mean" in n.lower()]
base_name = base_name[0] if base_name else res["valid_MAE"].idxmax()
imp = pd.DataFrame({"vs " + base_name: (1 - ref / ref[base_name]) * 100,
                    "vs next-best model": (1 - ref / ref.shift(1)) * 100}).round(1)
display(imp.rename_axis("model"))

## 4. Where do the models make their errors?
We load the data and the saved tuned XGBoost, predict the **test** year, and look at the errors by hour, weekday, month, station and station cluster. The seasonal-naive baseline (same hour last week) is added as a reference; it is not a deployable model because it needs last week's demand.

In [ ]:
import xgboost as xgb, joblib
df = load_demand()
mapping = load_cluster_mapping()
if mapping:
    df = add_cluster_feature(df, mapping)
train, valid, test = time_split(df)

meta = joblib.load("models/xgb_tuned_meta.pkl")
booster = xgb.Booster(); booster.load_model("models/xgb_tuned.json")
X = test.copy()
X["station"] = pd.Categorical(X["station"].astype(str), categories=meta["stations"]).codes   # same codes as in training
if "cluster" in meta["features"] and "cluster" not in X.columns:
    X = add_cluster_feature(X, meta["cluster_mapping"])
X = X[meta["features"]].astype("float32")
pred = booster.predict(xgb.DMatrix(X))
if meta["log_target"]:
    pred = np.expm1(pred)
test = test.assign(pred=np.clip(pred, 0, None))
test["err"] = test["pred"] - test[TARGET]
test["abs_err"] = test["err"].abs()

naive = seasonal_naive(df).reindex(test.index)
ok = naive.notna()
print("tuned XGBoost on test :", {k: round(v, 3) for k, v in regression_metrics(test[TARGET], test["pred"]).items()})
print("seasonal naive (same rows where last week exists):", {k: round(v, 3) for k, v in regression_metrics(test.loc[ok, TARGET], naive[ok]).items()})

In [ ]:
test["naive_abs_err"] = (naive - test[TARGET]).abs()
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
test.groupby("hour")[["abs_err", "naive_abs_err"]].mean().rename(columns={"abs_err": "tuned XGBoost", "naive_abs_err": "seasonal naive"}).plot(ax=ax[0], marker="o")
ax[0].set_title("MAE by hour of day"); ax[0].set_ylabel("MAE")
test.groupby("dayofweek")["abs_err"].mean().plot.bar(ax=ax[1]); ax[1].set_title("MAE by weekday (0 = Monday)")
test.groupby("month")["abs_err"].mean().plot.bar(ax=ax[2]); ax[2].set_title("MAE by month")
plt.tight_layout(); plt.show()

In [ ]:
# Bias: does the model over- or under-predict?
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
test.groupby("hour")["err"].mean().plot.bar(ax=ax[0], color="tab:purple"); ax[0].axhline(0, color="k", lw=0.8)
ax[0].set_title("Mean error by hour (positive = over-prediction)")
s = test.sample(min(len(test), 200_000), random_state=0)
ax[1].scatter(s[TARGET], s["pred"], s=2, alpha=0.15); lim = [0, s[TARGET].quantile(0.999)]
ax[1].plot(lim, lim, color="red", lw=1); ax[1].set_xlim(lim); ax[1].set_ylim(lim)
ax[1].set_xlabel("actual trips"); ax[1].set_ylabel("predicted trips"); ax[1].set_title("Predicted vs actual (test sample)")
plt.tight_layout(); plt.show()

In [ ]:
by_station = test.groupby("station", observed=True).agg(mae=("abs_err", "mean"), mean_trips=(TARGET, "mean"))
by_station["relative_mae"] = by_station["mae"] / by_station["mean_trips"]
print("Largest absolute error:"); display(by_station.sort_values("mae", ascending=False).head(10).round(2))
print("Largest error relative to the station's size:"); display(by_station.sort_values("relative_mae", ascending=False).head(10).round(2))
if "cluster" in test.columns:
    display(test.groupby("cluster")["abs_err"].agg(["mean", "count"]).rename(columns={"mean": "MAE"}).round(3))

In [ ]:
# The ten worst days (summed absolute error over all stations)
worst = test.assign(day=pd.to_datetime(test["ts"]).dt.date).groupby("day")["abs_err"].sum().sort_values(ascending=False).head(10)
display(worst.round(0).to_frame("total absolute error"))

## 5. Complexity versus accuracy
Training time, size and deployability matter as well as accuracy. Fill the last columns from the notebooks (training time is printed in 05 and 07).

In [ ]:
model_notes = pd.DataFrame({
    "model": res.index, "valid_MAE": res["valid_MAE"].round(3).values, "where": res["where"].values,
    "note": [str(n)[:90] for n in res.get("note", [""] * len(res))]})
display(model_notes.set_index("model"))

## 6. Conclusions
*(Write these in your own words after running everything. Prompts below.)*

1. **Which model wins on validation, and by how much** over the global mean, the lookup table and seasonal naive?
2. **Does test confirm it?** Is the validation-to-test gap small? Test and validation are different years, so some gap is expected.
3. **Why do trees beat the linear model (Ridge)?** Demand depends on interactions (station x hour x weekday).
4. **Boosting vs bagging vs AutoML:** what did FLAML's search show, and what did manual tuning of XGBoost add?
5. **What the error analysis shows:** which hours, months, stations or days are hardest, and is the model biased?
6. **Limits:** no weather, no past demand (the API cannot know it), only the top-100 stations, events and disruptions are unpredictable.
7. **Next steps:** weather features, a model per cluster, a better holiday or event calendar, blending models.